# 5. The flagging experiment: question, rule and the null results

**The question.** Can the smoothness of the calibration solutions guide
flagging? Gains of a well-behaved telescope vary slowly in time and frequency;
RFI that leaks into a solve makes them jump. The project's method flags a cell
only when its whitened residual is an outlier, and lets high gain total
variation (TV, notebook 3) lower the residual threshold locally:

$$z\,[1+\alpha\min(h/\operatorname{median}(h>0),5)]>\tau,\qquad z\geq2,$$

with $z$ the robust residual score, $h$ the TV guidance, $\tau \in \{8,5,3\}$ and
$\alpha \in \{0,1,3\}$ ($\alpha=0$ is the unguided control). Each candidate mask is
re-solved and scored by

$$L=\frac{T(G(F))}{T(G(F_0))}+2f+0.25\,\frac{\mathrm{MSE}_{\mathrm{val}}(F)}{\mathrm{MSE}_{\mathrm{val}}(F_0)},
\quad \mathrm{MSE}_{\mathrm{val}}(F)\leq1.01\,\mathrm{MSE}_{\mathrm{val}}(F_0),$$

with $f$ the fraction of training cells added; the no-change policy is always a
candidate. Test cells never choose anything.

**The decision rule** (frozen before every real-data run):

In [ ]:
%matplotlib inline
import os

os.environ.setdefault("RAYON_NUM_THREADS", "4")  # share the cores; set before the first core call

import numpy as np
import pandas as pd

from rangtoy.notebook import data, experiment, flags, gains, image, steer, survey

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

# Downloads and the extracted fixture go here; choose your own with RANG_NOTEBOOK_CACHE.
CACHE = data.default_cache_dir()
# LUNO sections run when a local copy of the slice exists (RANG_NOTEBOOK_LUNO=0 turns them off).
LUNO = os.environ.get("RANG_NOTEBOOK_LUNO", "1") != "0" and data.find_luno_slice() is not None
print("cache:", CACHE, "| LUNO slice available:", LUNO)
print(experiment.DECISION_RULE)

The protocol was frozen before the data were looked at; changing a threshold
after seeing a result would make the call meaningless. That is why the rule is
printed from the code, not restated.

## A replay on the fixture (exploratory, not a pre-registered result)

The cells below run the campaign's structure on the fixture: the original arm,
a comparator arm with a Tricolour strategy's additions, and the TV policy loop,
all with the frozen settings (2 × 32 bins, 50 iterations, Briggs 0.5, 512 ×
8″ images). It takes about 20 s.

In [ ]:
ds = data.load_fixture(cache_dir=CACHE)
pd.Series(data.summary(ds), name="value").to_frame()

In [ ]:
library = flags.strategy_library()
comparator = "caracal-calibrator-firstpass" if "caracal-calibrator-firstpass" in library else "teaching"
panel = steer.DecisionPanel(ds, library)  # the steering panel at the end reuses these results
arms, trials, images = panel.run(comparator)  # experiment.run_arms + experiment.image_arms
print("comparator strategy:", comparator)
experiment.arms_table(arms, images)

The TV loop's candidates. **What to look for:** which masks are admissible (no
lost gain support, validation MSE within 1 %), and whether any TV-guided mask
(tv weight 1 or 3) beats the no-change objective of 1.25.

In [ ]:
experiment.trials_table(trials)

In [ ]:
calls = experiment.decisions(arms, images)
display(calls)
experiment.ratio_figure(calls, f"fixture replay, comparator {comparator}")

**Reading it honestly.** On the fixture the TV policy selects no flags, so it
equals the original arm. The comparator's additions remove so many training
cells that some held-out cells lose their gain solution: the test MSE is
*undecidable*, not better or worse. A lower residual RMS for the policy against
a heavily flagged comparator mostly says the comparator imaged fewer cells.
This replay is a teaching aid on a pointing scan; the frozen results are below.

## The real results, from the archives

### First fixture campaign (2026-09, QuartiCal in a container)

In [ ]:
pd.Series(experiment.fixture_pilot(), name='fixture pilot').to_frame()

### LUNO PKS 1934−638, four stations (task K, 2026-10-30, on the core)

In [ ]:
display(pd.Series(experiment.archive_status("luno-four-station")).to_frame("task K"))
display(experiment.archive_arms("luno-four-station"))
k_calls = experiment.archive_hypotheses("luno-four-station")
display(k_calls)
experiment.ratio_figure(k_calls, "task K, frozen calls")

Null on every pair: the policy selects no flag, the first-pass strategy adds
128 cells to a slice already 70 % flagged and changes the test MSE by 6e-5 and
the residual RMS by 4e-5, relative. Cost comparable (ratio 1.30).

### LUNO PKS 1934−638, whole array (task L1)

In [ ]:
display(pd.Series(experiment.archive_status("luno-array")).to_frame("task L1"))
display(experiment.archive_arms("luno-array"))
l1_calls = experiment.archive_hypotheses("luno-array")
display(l1_calls)
experiment.ratio_figure(l1_calls, "task L1, frozen calls")

At array scale the frozen headline is **undecidable**: the first-pass strategy
removes the whole training support of some station-bins, so their held-out
cells cannot be predicted. Amendment 1, committed before the run, compares the
pair on the held-out cells both arms can predict. Its calls live in the run's
`result.json` (not in the compact archive); the cell shows them when the run
directory exists.

In [ ]:
matched = experiment.amendment_1_calls()
if matched is not None:
    display(matched)
    display(experiment.ratio_figure(matched, "task L1, amendment 1 matched-sample calls"))
else:
    print("run directory absent: see the table in research/luno-array-campaign.md")

## Reproduce task K's calls from the local slice

A result you can rerun is worth more than one you read. With a local copy of
the four-station slice, the replay below must give task K's ratios and
additions exactly (the core is deterministic). Skipped without the slice.

In [ ]:
if LUNO:
    luno = data.load_luno()
    strategy = library.get("caracal-calibrator-firstpass")
    if strategy is None:
        print("skipped: CARACAL first-pass strategy file not found")
    else:
        l_arms, l_trials = experiment.run_arms(luno, strategy)
        l_images = experiment.image_arms(luno, l_arms)
        replay = experiment.decisions(l_arms, l_images)
        display(replay)
        archived = k_calls.set_index(["candidate", "reference"])
        k_headline = archived.loc[("tv-policy", "tricolour-firstpass"), "mse_test ratio"]
        print("headline test-MSE ratio, replay vs archive:",
              replay.loc[0, "mse_test ratio"], k_headline)
else:
    print("skipped: no local LUNO slice")

## What the null result does and does not say

- It **does** say: on processed MeerKAT calibrator data at four stations and at
  array scale, gain-TV guidance selects nothing that improves held-out
  prediction or the point-source residual, and the CARACAL first pass does not
  either. Lower gain TV did not mean better prediction.
- It **does not** say the idea cannot work anywhere: the inputs were already
  60–70 % flagged with processed weights and an unlogged model about 1.5″ off
  centre; native flags (L3) and a second calibrator (L2, PKS 1830−211) are the
  open runs before the verdict (research/big-plan.md, wave 3).
- The held-out error floor may be **model-limited**: a point model of
  1934−638 leaves 0.2–0.5 % structured amplitude errors (Heywood et al. 2020,
  arXiv:2004.00454), and part of the gain structure that TV sees may be sky-model
  error rather than RFI or instrument. CARACAL ships a 196-component field
  model; a follow-up (L1-M) re-solves with it. The frozen arms keep the point
  model so they stay comparable with K and L1.
- A positive result would still need a matched full CARACAL comparison of
  quality *and* cost before it counted as an improvement.

## Steer it yourself

Choose another comparator strategy; the arms, loop, images and calls rerun.

In [ ]:
panel  # defaults to the comparator above, so it opens from the cache